# Treino com DistributedDataParallel (DDP)

Aula 3 - modelo inteiro em cada processo, dados fatiados. Vs `mp`: o DDP
abre a toolbox no `backward()`. O `all_reduce` dos gradientes faz os dois ranks andarem juntos.

1. cada rank pega o seu batch
2. forward + loss local
3. `backward()` local
4. DDP faz `all_reduce` dos gradientes
5. `optimizer.step()` - mesmo update nos dois

## Por que DDP existe

Uma GPU: epoch lento e batch pequeno. DDP = N cópias do modelo, 1/N dos dados,
`all_reduce` do gradiente. Resultado: **um** modelo, batch global `× N`.

O worker escolhe o device: CUDA se existir, senão CPU. 2 GPUs → `nccl` e
`cuda:rank`. 1 GPU / 2 processos → `gloo` na mesma GPU.

# 0. Instalando

`torchvision` só para o FashionMNIST.

In [17]:
%pip install -q torch torchvision

Note: you may need to restart the kernel to use updated packages.


## 1. Imports

`torch.utils.data.distributed.DistributedSampler` fatia os dados.
`torch.nn.parallel.DistributedDataParallel` sincroniza o gradiente.

In [18]:
import os

import torch
import torchvision


def pick_device(rank, world_size):
    """CUDA se existir; 2 processos compartilham a GPU se só houver uma."""
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        idx = rank if n >= world_size else 0
        torch.cuda.set_device(idx)
        return torch.device(f"cuda:{idx}")
    return torch.device("cpu")

## 2. Modelo

MLP 28×28 → 10 classes. Cada processo tem a sua cópia; o DDP deixa os pesos iguais.

In [19]:
class NeuralNetwork(torch.nn.Module):
    """MLP 28x28 -> 10 classes. O DDP deixa as cópias com os mesmos pesos."""
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(28 * 28, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)

## 3. Process group

Igual à toolbox. Porta `29512` para não colidir com o outro notebook.
`nccl` se cada rank tem a sua GPU; senão `gloo`.

In [20]:
def init(rank, world_size, port="29512"):
    """Entra no process group e devolve (device, backend). Porta distinta da toolbox."""
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = port
    device = pick_device(rank, world_size)
    n_gpu = torch.cuda.device_count() if device.type == "cuda" else 0
    backend = "nccl" if n_gpu >= world_size else "gloo"
    torch.distributed.init_process_group(backend=backend, rank=rank, world_size=world_size)
    return device, backend

## 4. DistributedSampler

Cada rank vê 1/N. `batch=32` é por rank (global = `32 * N`).
`barrier()`: só o rank 0 baixa o dataset.

In [21]:
def get_loader(rank, world_size, n_samples=2048):
    """DistributedSampler no recorte. 60k × 3 épocas também deixa o notebook lento."""
    if rank == 0:
        torchvision.datasets.FashionMNIST(root="data", train=True, download=True)
    torch.distributed.barrier()

    train_data = torchvision.datasets.FashionMNIST(
        root="data",
        train=True,
        download=False,
        transform=torchvision.transforms.ToTensor(),
    )
    train_data = torch.utils.data.Subset(train_data, range(n_samples))
    sampler = torch.utils.data.distributed.DistributedSampler(
        train_data, num_replicas=world_size, rank=rank
    )
    loader = torch.utils.data.DataLoader(train_data, batch_size=32, sampler=sampler)
    return loader, sampler

## 5. DDP

`torch.nn.parallel.DistributedDataParallel(model)` registra hooks: `backward()` local,
depois `all_reduce` do gradiente. Em CUDA, `device_ids=[device.index]`.

In [22]:
def build_model(device):
    """Wrap DDP: broadcast dos pesos + hooks de all_reduce no backward."""
    net = NeuralNetwork().to(device)
    kwargs = {"device_ids": [device.index]} if device.type == "cuda" else {}
    model = torch.nn.parallel.DistributedDataParallel(net, **kwargs)
    opt = torch.optim.SGD(model.parameters(), lr=0.001)
    loss_f = torch.nn.CrossEntropyLoss()
    return model, opt, loss_f

## 6. Loop de treino

Igual ao single-GPU. `sampler.set_epoch(epoch)` a cada época (senão a fatia trava).

In [23]:
def train_loop(rank, loader, sampler, model, opt, loss_f, device, epochs=3):
    """Loop normal. A sincronização está escondida no backward do DDP."""
    for epoch in range(epochs):
        sampler.set_epoch(epoch)
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_f(model(x), y)
            loss.backward()
            opt.step()
        if rank == 0:
            print(f"epoch {epoch}  loss={loss.item():.4f}")

## 7. Rodar com 2 processos

Grupo → dataset → DDP → treinar → `torch.distributed.destroy_process_group`. Rode por último.
Recorte de 2048 imagens: o FashionMNIST inteiro × 3 épocas demora na aula.

In [24]:
def worker(rank, world_size):
    """Main de cada rank: grupo, dados, DDP, treino, fecha o grupo."""
    device, backend = init(rank, world_size)
    if rank == 0:
        print(f"backend={backend}  device={device}")
    loader, sampler = get_loader(rank, world_size)
    model, opt, loss_f = build_model(device)
    train_loop(rank, loader, sampler, model, opt, loss_f, device)
    torch.distributed.destroy_process_group()

world_size = 2
torch.multiprocessing.start_processes(
    worker, args=(world_size,), nprocs=world_size, join=True, start_method="fork"
)

backend=gloo  device=cuda:0
epoch 0  loss=2.2960
epoch 1  loss=2.3078
epoch 2  loss=2.2773
